In [0]:
from pyspark.sql import functions as F

# Schemas für Silver & Gold anlegen
spark.sql("CREATE SCHEMA IF NOT EXISTS security_lakehouse.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS security_lakehouse.gold")

# Alle JSONL-Logs aus der Landing Zone einlesen
raw_df = (
    spark.read.json("/Volumes/security_lakehouse/bronze/landing_zone/*.jsonl")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

raw_df.write.format("delta").mode("overwrite").saveAsTable(
    "security_lakehouse.bronze.raw_security_events"
)

print(f"Bronze geladen: {raw_df.count()} Events")
display(raw_df.limit(5))

Bronze geladen: 272 Events


asn,asn_org,category,event_id,geo_country,operation_name,source_ip,status_code,target_role,timestamp,user_principal_name,_ingested_at,_source_file
AS3209,Vodafone GmbH,ActivityLogs,690661f3-9779-44ae-a764-5ad111aa223a,DE,Microsoft.Compute/virtualMachines/read,84.152.164.230,0,null,2026-11-20T18:30:55.663200+00:00,anna.weber@sec-lab.de,2026-11-21T02:26:29.344Z,dbfs:/Volumes/security_lakehouse/bronze/landing_zone/azure_security_logs_20261121_022255.jsonl
AS3209,Vodafone GmbH,ActivityLogs,659b53f2-ec97-40fd-b89b-62485cfa52d2,DE,Microsoft.Storage/storageAccounts/blobServices/containers/read,84.152.152.5,0,null,2026-11-20T15:36:55.663200+00:00,anna.weber@sec-lab.de,2026-11-21T02:26:29.344Z,dbfs:/Volumes/security_lakehouse/bronze/landing_zone/azure_security_logs_20261121_022255.jsonl
AS8075,Microsoft Corporation,ActivityLogs,f6b0c956-e273-4178-a08d-396b6bcb6330,DE,Microsoft.Storage/storageAccounts/blobServices/containers/read,84.152.21.140,0,null,2026-11-20T09:25:55.663200+00:00,svc-deploy-pipeline@sec-lab.de,2026-11-21T02:26:29.344Z,dbfs:/Volumes/security_lakehouse/bronze/landing_zone/azure_security_logs_20261121_022255.jsonl
AS8075,Microsoft Corporation,ActivityLogs,3e98db6a-1325-44fe-b430-49748abf2159,DE,Microsoft.Storage/storageAccounts/blobServices/containers/read,84.152.216.132,0,null,2026-11-20T14:06:55.663200+00:00,svc-deploy-pipeline@sec-lab.de,2026-11-21T02:26:29.344Z,dbfs:/Volumes/security_lakehouse/bronze/landing_zone/azure_security_logs_20261121_022255.jsonl
AS3209,Vodafone GmbH,ActivityLogs,c003e05b-ad62-4202-8bac-1a23ac0698a0,DE,Microsoft.Storage/storageAccounts/blobServices/containers/read,84.152.49.3,0,null,2026-11-20T15:09:55.663200+00:00,anna.weber@sec-lab.de,2026-11-21T02:26:29.344Z,dbfs:/Volumes/security_lakehouse/bronze/landing_zone/azure_security_logs_20261121_022255.jsonl


In [0]:
bronze_df = spark.table("security_lakehouse.bronze.raw_security_events")

silver_df = (
    bronze_df.dropDuplicates(["event_id"])
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .withColumn("event_hour", F.hour("event_ts"))
    .withColumn(
        "status_label",
        F.when(F.col("status_code") == 0, "SUCCESS")
        .when(F.col("status_code") == 50126, "INVALID_PASSWORD")
        .when(F.col("status_code") == 403, "FORBIDDEN")
        .otherwise("OTHER"),
    )
    .withColumn("is_off_hours", (F.col("event_hour") < 6) | (F.col("event_hour") > 20))
    .withColumn("is_foreign_country", F.col("geo_country") != "DE")
    .drop("timestamp")
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(
    "security_lakehouse.silver.clean_security_events"
)

print(f"Silver bereinigt: {silver_df.count()} eindeutige Events")

Silver bereinigt: 272 eindeutige Events


In [0]:
from pyspark.sql.window import Window

silver = spark.table("security_lakehouse.silver.clean_security_events")

# 1. Detection: Password Spraying & Brute Force (MITRE T1110)
spray_alerts = (
    silver.filter((F.col("category") == "SignInLogs") & (F.col("status_code") == 50126))
    .groupBy("user_principal_name", "geo_country", "asn_org")
    .agg(
        F.count("*").alias("failed_attempts"),
        F.countDistinct("source_ip").alias("distinct_ips"),
        F.max("event_ts").alias("detected_at"),
    )
    .filter(F.col("failed_attempts") >= 5)
    .select(
        F.expr("uuid()").alias("alert_id"),
        "detected_at",
        F.lit("HIGH").alias("severity"),
        F.lit("Credential Access (T1110)").alias("mitre_tactic"),
        F.lit("Password Spraying / Brute Force").alias("rule_name"),
        "user_principal_name",
        F.concat(F.col("distinct_ips"), F.lit(" rotating IPs")).alias("source_ip"),
        "geo_country",
        F.concat(
            F.col("failed_attempts"),
            F.lit(" fehlgeschlagene Logins über "),
            F.col("asn_org"),
        ).alias("details"),
    )
)

# 2. Detection: Impossible Travel (MITRE T1078)
user_win = Window.partitionBy("user_principal_name").orderBy("event_ts")
travel_alerts = (
    silver.filter((F.col("category") == "SignInLogs") & (F.col("status_code") == 0))
    .withColumn("prev_country", F.lag("geo_country").over(user_win))
    .withColumn("prev_ts", F.lag("event_ts").over(user_win))
    .withColumn(
        "mins_diff",
        (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_ts")) / 60.0,
    )
    .filter(
        (F.col("prev_country").isNotNull())
        & (F.col("geo_country") != F.col("prev_country"))
        & (F.col("mins_diff") <= 60)
    )
    .select(
        F.expr("uuid()").alias("alert_id"),
        F.col("event_ts").alias("detected_at"),
        F.lit("CRITICAL").alias("severity"),
        F.lit("Initial Access (T1078)").alias("mitre_tactic"),
        F.lit("Impossible Travel").alias("rule_name"),
        "user_principal_name",
        "source_ip",
        "geo_country",
        F.concat(
            F.lit("Login-Wechsel "),
            F.col("prev_country"),
            F.lit(" -> "),
            F.col("geo_country"),
            F.lit(" in "),
            F.round("mins_diff", 1),
            F.lit(" Min."),
        ).alias("details"),
    )
)

# 3. Detection: Off-Hours Privilege Escalation (MITRE T1098)
privesc_alerts = (
    silver.filter(
        (F.col("operation_name") == "Microsoft.Authorization/roleAssignments/write")
        & (F.col("target_role").isin("Owner", "Contributor", "User Access Administrator"))
    )
    .select(
        F.expr("uuid()").alias("alert_id"),
        F.col("event_ts").alias("detected_at"),
        F.lit("CRITICAL").alias("severity"),
        F.lit("Privilege Escalation (T1098)").alias("mitre_tactic"),
        F.lit("Privileged Role Assignment").alias("rule_name"),
        "user_principal_name",
        "source_ip",
        "geo_country",
        F.concat(
            F.lit("Rolle '"),
            F.col("target_role"),
            F.lit("' vergeben über "),
            F.col("asn_org"),
        ).alias("details"),
    )
)

gold_alerts = spray_alerts.unionByName(travel_alerts).unionByName(privesc_alerts)
gold_alerts.write.format("delta").mode("overwrite").saveAsTable(
    "security_lakehouse.gold.security_alerts"
)

display(gold_alerts.orderBy(F.desc("detected_at")))

alert_id,detected_at,severity,mitre_tactic,rule_name,user_principal_name,source_ip,geo_country,details
07b25f8d-bc17-4f20-8050-8c8c83ae3808,2026-11-21T03:14:55.663Z,CRITICAL,Privilege Escalation (T1098),Privileged Role Assignment,lukas.meyer@sec-lab.de,167.99.142.11,US,Rolle 'Owner' vergeben über DigitalOcean LLC
d27b74b9-8a8e-4f7b-9099-64a39afca3e1,2026-11-21T02:04:55.663Z,CRITICAL,Initial Access (T1078),Impossible Travel,lukas.meyer@sec-lab.de,167.99.142.11,US,Login-Wechsel DE -> US in 12.0 Min.
a742cd3f-5b9f-4cb2-bf48-5630b5a4f285,2026-11-21T01:42:10.663Z,HIGH,Credential Access (T1110),Password Spraying / Brute Force,anna.weber@sec-lab.de,18 rotating IPs,RU,18 fehlgeschlagene Logins über Rostelecom
